# 120 — Human-in-the-loop y aprobaciones

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**HITL:** ciertos pasos requieren decisión humana EN tiempo de ejecución (distinto de
on-the-loop: supervisión a posteriori). El `ask` de la matriz de permisos (119) se
materializa con el patrón **interrupt/resume**: el runtime suspende el bucle en un
punto consistente, persiste un checkpoint (118) con la acción propuesta y su
justificación, espera el veredicto (minutos u horas — el proceso puede morir y
renacer) y reanuda: **aprobar** (ejecutar), **editar** (ejecutar la versión
corregida) o **rechazar** (el motivo entra al contexto como observación y el agente
replantea).

### 📋 Contrato de la solicitud y calibración

El aprobador debe ver: (1) la acción EXACTA con argumentos literales, (2) el porqué
(objetivo + observaciones), (3) el impacto (clase de efecto, alcance, dry-run cuando
exista), (4) las alternativas si se rechaza. Si no puede decidir con eso, el defecto
es de la solicitud.

Criterio económico: `ask` se justifica cuando costo_error × prob_error > costo de
atención. Antídotos a la fatiga: aprobar por lotes/planes, umbrales cuantitativos,
y medir la tasa de rechazo (100 % de aprobaciones durante un mes = punto mal
calibrado). El laboratorio `workflow` ejecuta el esqueleto: `waiting_approval` es el
interrupt; `approved: true`, el veredicto; `completed` solo existe después.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Interrupt: `validated → waiting_approval` (el flujo se detiene en un
estado de espera persistente). Resume: `waiting_approval → completed`, que solo existe
porque `approved: true` quedó registrado antes — con `approved: false` esa transición
sería ilegítima. Estado nuevo: `rejected` (terminal, con campo `motivo`), o mejor
`waiting_approval → replanning` para que el rechazo vuelva al agente como observación
y el plan se revise (115) en lugar de morir en silencio.

**Ejercicio 2.** Solicitud válida — Acción: `delete_branch(repo="app",
branch="release-2024")`, argumentos literales. Porqué: sin commits desde 2025-01;
ninguna referencia desde CI ni despliegues; objetivo: higiene del repositorio.
Impacto: irreversible sin respaldo — dry-run adjunto: "se eliminarían 214 commits no
presentes en main; 2 tags apuntan a esta rama". Alternativas: rechazar (la rama
queda); editar (archivar como tag `archive/release-2024` antes de borrar — plan B
recomendado). Nótese que el dry-run transforma la decisión: los "2 tags" son
exactamente lo que un aprobador necesita ver.

**Ejercicio 3.** `read_inbox` pura → allow. `draft_reply` reversible → allow.
`send_reply` irreversible/externa → ask… pero 80/día produce fatiga: pasar a
aprobación por lote (revisar la cola 2 veces al día) o auto-allow con auditoría para
respuestas de plantilla y ask solo para redacción libre. `add_to_blocklist`
reversible pero con impacto en clientes → ask. `refund ≤ 30` → allow con presupuesto
diario acumulado (121). `refund > 30` → ask con identidad fuerte del aprobador.

**Ejercicio 4.** Ver celda: los tres veredictos producen tres caminos distintos y el
log registra quién, qué versión y por qué — el rechazo NO lanza excepción: devuelve el
motivo como observación del paso.

In [ ]:
result = run_lab("workflow", seed=120)
assert result["kind"] == "workflow"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — interrupt/resume localizados y verificados
result = run_lab("workflow", seed=120)
events = result["result"]["events"]
interrupt = {"from": "validated", "to": "waiting_approval"}
resume = {"from": "waiting_approval", "to": "completed"}
assert interrupt in events and resume in events
assert result["result"]["approved"] is True
print("interrupt:", interrupt)
print("resume  :", resume, "(legitimo solo porque approved=True)")


In [ ]:
# Ejercicio 4 — interrupt/resume con los tres veredictos
def ejecutar_plan(pasos, aprobador):
    log = []
    for paso in pasos:
        if paso["politica"] == "allow":
            log.append({"accion": paso["accion"], "resultado": "ejecutada", "via": "allow"})
            continue
        veredicto = aprobador(paso)                     # INTERRUPT -> veredicto
        if veredicto["verdict"] == "aprobar":
            log.append({"accion": paso["accion"], "resultado": "ejecutada",
                        "via": "ask/aprobada", "approver": veredicto["quien"]})
        elif veredicto["verdict"] == "editar":
            log.append({"accion": veredicto["edit"], "resultado": "ejecutada",
                        "via": "ask/editada", "original": paso["accion"],
                        "approver": veredicto["quien"]})
        else:
            log.append({"accion": paso["accion"], "resultado": "rechazada",
                        "motivo": veredicto["motivo"], "approver": veredicto["quien"]})
    return log

VEREDICTOS = {"enviar_aviso": {"verdict": "editar", "quien": "mgarcia",
                               "edit": "enviar_aviso(excluye_premium=True)"},
              "borrar_rama": {"verdict": "rechazar", "quien": "mgarcia",
                              "motivo": "2 tags apuntan a la rama"},
              "publicar_nota": {"verdict": "aprobar", "quien": "mgarcia"}}

plan = [{"accion": "generar_borrador", "politica": "allow"},
        {"accion": "enviar_aviso", "politica": "ask"},
        {"accion": "borrar_rama", "politica": "ask"},
        {"accion": "publicar_nota", "politica": "ask"}]

for linea in ejecutar_plan(plan, lambda p: VEREDICTOS[p["accion"]]):
    print(linea)


## Reflexión

1. En el laboratorio la aprobación es síncrona e instantánea. ¿Qué dos piezas de
   ingeniería (clases 115 y 116) se vuelven imprescindibles cuando el veredicto tarda
   horas, y por qué?
2. ¿Por qué el veredicto "editar" evita un falso dilema, y qué debe registrar el log
   para que esa edición sea auditable?
3. Tu punto de aprobación lleva 3 meses con 100 % de aprobaciones. Da las dos lecturas
   posibles de esa métrica y qué acción tomarías en cada caso.